# Side-by-side: UNet vs image-domain I2SB vs latent I2SB

Runs all methods on the **same** batch and stacks them against the T1ce ground truth:
1. **UNet (yhat)** -- the e2e synthesis output.
2. **image I2SB (T1)** -- the T1-prior regressor (`I2SB_T1_CFG`), sampled from T1.
3. **I2SB from yhat** -- a SEPARATELY trained yhat-prior regressor (`I2SB_YHAT_CFG`), sampled from yhat.
4. **(optional) Latent I2SB** -- design-A pure-latent sampler (`RUN_LATENT`).

Each i2sb column loads its OWN net + bridge/tau. Metrics map the data from ~[-1,1] to [0,1] (add 1,
/2) before PSNR/SSIM. **Prereqs (HPC):** each run's saved `config.json` + `net.ckpt`, and the synth
run. Edit the `os.chdir` path. Any i2sb config left empty / missing is skipped.

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

os.chdir('/scratch/ee2178/ImMAP')     # <-- EDIT to your repo root

# ---- pick your runs (SAVED config.json paths); the two i2sb regressors are DIFFERENT nets ----
SYNTH_CFG       = "trained_nets/brats/Synth_T1ce_Pretrain_VGG_CosLR/config.json"   # UNet -> yhat
I2SB_T1_CFG     = "trained_nets/brats/I2SB_T1ce_from_T1/config.json"               # T1-prior regressor
I2SB_YHAT_CFG   = "trained_nets/brats/I2SB_T1ce_from_Synth/config.json"            # yhat-prior regressor (separate net)
LATENT_I2SB_CFG = "trained_nets/brats/Latent_I2SB_T1ce/config.json"               # latent run
RUN_LATENT      = True          # set False to skip the (currently poor) latent method
SPLIT, NFE, USE_EMA = "val", 100, True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from models import build_model
from training.common import load_model, apply_loss_mask
from training.metrics import compute_metrics
from sb.base import build_schedule, bridge_coeffs
from sb.i2sb import i2sb_sample
import datasets                                   # registers loaders
from datasets.registry import build_loader

# build_model + load net.ckpt (+ EMA shadow) + compile flex; returns (net, cfg)
def load_net_ema(config_path, use_ema=USE_EMA):
    with open(config_path) as f:
        c = json.load(f)
    sd = os.path.dirname(config_path)
    net = build_model(c).to(device)
    net.load_state_dict(torch.load(os.path.join(sd, "net.ckpt"), map_location=device,
                                   weights_only=False)["model_state_dict"])
    net.eval()
    ema_p = os.path.join(sd, "ema.pt")
    if use_ema and os.path.exists(ema_p):
        shadow = torch.load(ema_p, map_location=device, weights_only=False)["shadow"]
        for p, s in zip([p for p in net.parameters() if p.requires_grad], shadow):
            p.data.copy_(s.to(p.device))
    if getattr(net, "attn_backend", None) == "flex":
        net.compile_flex()
    return net, c

# an i2sb run: build its net + its own bridge from its config
def load_i2sb_run(config_path):
    net, cfg = load_net_ema(config_path)
    ic = cfg["i2sb"]
    bridge = build_schedule(kind=ic.get("kind", "brownian"), n_points=ic.get("n_points", 1000),
                          device=device, tau=ic.get("tau", 0.19),                          beta_max=ic.get("beta_max", 0.3))
    return net, bridge, cfg

def sample_i2sb(net, bridge, cfg, x1b, cb):
    ic = cfg["i2sb"]
    cbb = cb if int(cfg["model"]["params"]["C"]) > 1 else None     # condition only if the net expects it
    r, _, _ = i2sb_sample(net, x1b, bridge, cond=cbb, nfe=NFE, deterministic=ic.get("deterministic", False),
                          clip_denoise=ic.get("clip_denoise", False),
                          target_channels=ic.get("target_channels", 1), verbose=False)
    return r

# data is scaled to ~[-1,1]; map to [0,1] (add 1, /2) BEFORE masking so PSNR/SSIM see the right range
def metrics01(gt, pred, m, psnr_only=True):
    g, r = apply_loss_mask((gt + 1) / 2, (pred + 1) / 2, m, True)
    return compute_metrics(g, r, psnr_only=psnr_only)

print("device:", device)

## 1. One shared batch + the UNet output (yhat)

A single canonical loader (x1=T1, cond=[FLAIR,T1,T2], fixed 192 crop) gives the batch every method sees. `yhat` is computed from the synth net on `cond` (no precompute needed).

In [ ]:
# canonical loader: x1 = T1, cond = [FLAIR,T1,T2], fixed 192 crop
ref_cfg = json.load(open(I2SB_YHAT_CFG if os.path.exists(I2SB_YHAT_CFG) else I2SB_T1_CFG))
base = dict(ref_cfg["data"][SPLIT])
base.update(name="i2sb", x1_source="contrast", cond_idx=[0, 1, 3], center_crop=192,
            random_flips=False, num_workers=0)
base.pop("crop_size", None)
loader = build_loader(base, shuffle=True, drop_last=False)
x0, x1_t1, cond, mask = next(iter(loader))          # x0=T1ce GT, x1_t1=T1 prior, cond=[FLAIR,T1,T2]
x0, x1_t1, mask, cond = x0.to(device), x1_t1.to(device), mask.to(device), cond.to(device)

# yhat = UNet(cond): undo the i2sb /scales, run the synth net (pad to 2^num_pool), re-apply x0 scale
synth = load_model(SYNTH_CFG, device=device); synth.eval()
scfg = json.load(open(SYNTH_CFG))
scales = torch.tensor(base["scales"], device=device)
cidx, x0i = list(base["cond_idx"]), int(base["x0_idx"])
mult = 2 ** int(scfg["model"]["params"].get("num_pool_layers", 5))
assert list(scfg["data"]["train"]["input_idx"]) == cidx, "synth input_idx != cond_idx"

def compute_yhat(cb):
    with torch.no_grad():
        s = cb * scales[cidx].view(1, -1, 1, 1)
        h, w = s.shape[-2:]; a, b = (-h) % mult, (-w) % mult
        if a or b:
            s = F.pad(s, (0, b, 0, a), mode="replicate")
        return synth(s)[..., :h, :w] / scales[x0i]

yhat = compute_yhat(cond)
print("batch:", tuple(x0.shape), "| yhat:", tuple(yhat.shape))

## 2. The two image-domain I2SB regressors (separate nets)

`I2SB_T1_CFG` (T1-prior) sampled from **T1**, and `I2SB_YHAT_CFG` (a different, yhat-prior net) sampled from **yhat** -- each with its own bridge/tau and conditioning.

In [ ]:
net_t1 = bridge_t1 = cfg_t1 = recon_t1 = None
net_yh = bridge_yh = cfg_yh = recon_yh = None

if I2SB_T1_CFG and os.path.exists(I2SB_T1_CFG):
    net_t1, bridge_t1, cfg_t1 = load_i2sb_run(I2SB_T1_CFG)
    recon_t1 = sample_i2sb(net_t1, bridge_t1, cfg_t1, x1_t1, cond)       # T1-prior net, from T1
    print("image I2SB (T1)  done | C =", cfg_t1["model"]["params"]["C"], "| tau =", cfg_t1["i2sb"].get("tau"))

if I2SB_YHAT_CFG and os.path.exists(I2SB_YHAT_CFG):
    net_yh, bridge_yh, cfg_yh = load_i2sb_run(I2SB_YHAT_CFG)
    recon_yh = sample_i2sb(net_yh, bridge_yh, cfg_yh, yhat, cond)        # yhat-prior net, from yhat
    print("I2SB from yhat   done | C =", cfg_yh["model"]["params"]["C"], "| tau =", cfg_yh["i2sb"].get("tau"))

## 3. (optional) Latent I2SB

Design-A pure-latent sampler. Skipped if `RUN_LATENT=False`.

In [ ]:
recon_lat = None
if RUN_LATENT:
    from training.latent_i2sb import _load_frozen_dict
    from sb.latent_i2sb import latent_i2sb_sample
    R, cfg_lat = load_net_ema(LATENT_I2SB_CFG)
    li, ld = cfg_lat["i2sb"], cfg_lat["dicts"]
    D_joint = _load_frozen_dict(ld["joint_dict_config"], device, backend=ld.get("dict_backend"))
    D_t1ce  = _load_frozen_dict(ld["t1ce_dict_config"],  device, backend=ld.get("dict_backend"))
    bridge_lat = build_schedule(kind=li.get("kind", "brownian"), n_points=li.get("n_points", 1000),
                              device=device, tau=li.get("tau", 0.024),
                              beta_max=li.get("beta_max", 0.3))

    def run_latent(x1b, cb):
        return latent_i2sb_sample(D_joint, R, D_t1ce, x1b, cb, bridge_lat,
                                  conditional=li.get("conditional", True), M=D_joint.M, nfe=NFE,
                                  deterministic=li.get("deterministic", False), decode_dc=li.get("decode_dc", "x1_mean"))

    recon_lat = run_latent(x1_t1, cond)
    print("latent I2SB done  (tau =", li.get("tau"), ")")
else:
    print("latent I2SB skipped")

## 4. Side by side

T1 | UNet(yhat) | image I2SB (T1) | I2SB from yhat | (latent) | GT, with masked PSNR (computed on the [0,1]-mapped images) under each. Missing methods are dropped.

In [ ]:
def _win(*imgs, m):
    v = torch.cat([im[m.bool().expand_as(im)].flatten() for im in imgs]).float().cpu()
    return float(v.quantile(0.01)), float(v.quantile(0.99))

methods = [("T1 prior", x1_t1), ("UNet (yhat)", yhat)]
if recon_t1 is not None:  methods.append(("image I2SB (T1)", recon_t1))
if recon_yh is not None:  methods.append(("I2SB from yhat", recon_yh))
if recon_lat is not None: methods.append(("latent I2SB", recon_lat))
methods.append(("T1ce GT", x0))

nshow = min(4, x0.shape[0])
vmn, vmx = _win(x1_t1[:nshow], x0[:nshow], m=mask[:nshow])
ncol = len(methods)
fig, ax = plt.subplots(nshow, ncol, figsize=(2.15 * ncol, 2.5 * nshow)); ax = np.atleast_2d(ax)
for i in range(nshow):
    for j, (name, img) in enumerate(methods):
        disp = (img[i, 0] * mask[i, 0]).detach().cpu().numpy()
        if name == "T1ce GT":
            ttl = name
        else:
            ttl = f"{name}\n{float(metrics01(x0[i:i+1], img[i:i+1], mask[i:i+1], psnr_only=True)['psnr']):.1f} dB"
        ax[i, j].imshow(disp, cmap="gray", vmin=vmn, vmax=vmx)
        ax[i, j].set_title(ttl, fontsize=8); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
plt.tight_layout(); plt.show()

## 5. (optional) Aggregate metrics over a few batches

Re-runs each present method on `N_BATCHES` fresh batches (samplers are the cost -- keep it small) and reports mean masked PSNR / SSIM (on the [0,1]-mapped images).

In [ ]:
N_BATCHES = 3

agg = {"UNet (yhat)": []}
if recon_t1 is not None:  agg["image I2SB (T1)"] = []
if recon_yh is not None:  agg["I2SB from yhat"] = []
if recon_lat is not None: agg["latent I2SB"] = []
it = iter(loader)
for _ in range(N_BATCHES):
    try:
        a0, a1, ac, am = next(it)
    except StopIteration:
        break
    a0, a1, am, ac = a0.to(device), a1.to(device), am.to(device), ac.to(device)
    yh = compute_yhat(ac)
    outs = {"UNet (yhat)": yh}
    if recon_t1 is not None:  outs["image I2SB (T1)"] = sample_i2sb(net_t1, bridge_t1, cfg_t1, a1, ac)
    if recon_yh is not None:  outs["I2SB from yhat"]  = sample_i2sb(net_yh, bridge_yh, cfg_yh, yh, ac)
    if recon_lat is not None: outs["latent I2SB"]     = run_latent(a1, ac)
    for name, out in outs.items():
        mets = metrics01(a0, out, am, psnr_only=False)
        agg[name].append((float(mets["psnr"]), float(mets["ssim"])))

nb = len(next(iter(agg.values())))
print(f"{'method':16s} {'PSNR':>8} {'SSIM':>8}   (mean over {nb} batches)")
for name, vals in agg.items():
    print(f"{name:16s} {np.mean([v[0] for v in vals]):8.2f} {np.mean([v[1] for v in vals]):8.3f}")